# Investigator Agent Experiments

This notebook reuses the existing Python modules so the target model is loaded only once per kernel session.

Edit these between runs:
- prompts (`PROMPT_CONFIG`)
- tool-call defaults (`TOOL_DEFAULTS`)
- model choices in `RUN_CONFIG`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List

from openai import OpenAI

import investigator_agent_loop as ial
import query_target_model_roles as qtmr
from prompts import (
    AUDITING_SYSTEM_PROMPT_MULTIPLE_QUIRKS,
    INVESTIGATOR_AGENT_PROMPT_MULTIPLE_QUIRKS_ASSISTANT_SAMPLING_WITH_MEMORY,
    INVESTIGATOR_AGENT_PROMPT_MULTIPLE_QUIRKS_USER_SAMPLING_WITH_MEMORY,
)



In [ ]:
# Edit these for each experiment run
RUN_CONFIG = {
    "run_dir": Path("agent_runs"),
    "target_config": Path("config.ini"),
    "investigator_model": "gpt-5",
    "aux_model": "gpt-4",
    "max_output_tokens": 16000,
    "parallel_tool_calls": False,
    "sample_role": "assistant",  # or "user"
}

if RUN_CONFIG["sample_role"] == "assistant":
    investigator_prompt = INVESTIGATOR_AGENT_PROMPT_MULTIPLE_QUIRKS_ASSISTANT_SAMPLING_WITH_MEMORY
else:
    investigator_prompt = INVESTIGATOR_AGENT_PROMPT_MULTIPLE_QUIRKS_USER_SAMPLING_WITH_MEMORY

PROMPT_CONFIG = {
    "system_prompt": AUDITING_SYSTEM_PROMPT_MULTIPLE_QUIRKS,
    "investigator_user_prompt": investigator_prompt,
}

# Fallback defaults used when the model omits tool args.
TOOL_DEFAULTS = {
    "k": 1,
    "validate": False,
    "samples_per_role": 1,
    "max_new_tokens": 220,
    "min_new_tokens": 100,
    "temperature": 0.9,
    "top_p": 0.95,
    "repetition_penalty": 1.08,
    "stop_on_eot": True,
    "max_items": 400,
    "max_chars": 180_000,
    "max_sample_chars": 1500,
    "sample_index": 1,
}



In [ ]:
# Load target model ONCE per kernel. Re-run this cell only if you change model/config.
def load_target_model_once(target_config: Path) -> Dict[str, Any]:
    qtmr.DEFAULT_CONFIG_PATH = target_config
    model_id, adapter_id, hf_cache_dir, allow_download = qtmr.load_target_model_config()
    hf_cache_dir = qtmr.configure_hf_cache(hf_cache_dir)
    device, dtype, backend = qtmr.detect_device()
    qtmr.device, qtmr.tok, qtmr.model = device, *qtmr.setup_model(
        device,
        dtype,
        model_id=model_id,
        adapter_id=adapter_id,
        hf_cache_dir=hf_cache_dir,
        allow_download=allow_download,
    )
    info = {
        "backend": backend,
        "device": str(device),
        "dtype": str(dtype),
        "model_id": model_id,
        "adapter_id": adapter_id,
        "hf_cache_dir": str(hf_cache_dir),
        "allow_download": allow_download,
    }
    return info

TARGET_INFO = load_target_model_once(RUN_CONFIG["target_config"])
TARGET_INFO


In [ ]:
def _append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, ensure_ascii=False) + "\n")


def start_session(
    run_config: Dict[str, Any],
    prompt_config: Dict[str, str],
    tool_defaults: Dict[str, Any],
    tools: List[Dict[str, Any]] | None = None,
) -> Dict[str, Any]:
    """Start one investigator run (same loop mechanics as investigator_agent_loop.py)."""
    ial._ensure_api_key()
    run_dir = ial._make_run_dir(run_config["run_dir"])
    probe_runs_root = run_dir / "probe_runs"
    probe_runs_root.mkdir(parents=True, exist_ok=True)

    memory_path = run_dir / "memory.md"
    memory_events_path = run_dir / "memory_events.jsonl"
    ial._init_memory_file(memory_path)

    client = OpenAI()
    tools = tools if tools is not None else ial.build_tools_spec(run_config["sample_role"])

    investigator_prompt = prompt_config["investigator_user_prompt"]
    last_assistant_items: List[Dict[str, Any]] = []
    last_function_call_items: List[Dict[str, Any]] = []
    last_tool_outputs: List[Dict[str, Any]] = []

    context = ial.build_input_context(
        investigator_prompt,
        ial._read_memory(memory_path),
        last_assistant_items=last_assistant_items,
        last_function_call_items=last_function_call_items,
        last_tool_outputs=last_tool_outputs,
    )

    response = client.responses.create(
        model=run_config["investigator_model"],
        instructions=prompt_config["system_prompt"],
        input=context,
        tools=tools,
        tool_choice="auto",
        max_output_tokens=run_config["max_output_tokens"],
        parallel_tool_calls=run_config["parallel_tool_calls"],
    )

    history_path = run_dir / "conversation.jsonl"
    tool_calls_path = run_dir / "tool_calls.jsonl"
    _append_jsonl(history_path, ial._response_to_loggable(response))

    response_text = ial._extract_output_text(response)
    cleaned_response_text, updates = ial.extract_memory_updates(response_text)
    ial.append_memory(
        memory_path,
        updates,
        0,
        memory_events_path=memory_events_path,
        model=run_config["investigator_model"],
    )
    last_assistant_items = ial._latest_assistant_message_items(response)

    return {
        "run_dir": run_dir,
        "probe_runs_root": probe_runs_root,
        "memory_path": memory_path,
        "memory_events_path": memory_events_path,
        "client": client,
        "tools": tools,
        "response": response,
        "cleaned_response_text": cleaned_response_text,
        "history_path": history_path,
        "tool_calls_path": tool_calls_path,
        "run_config": dict(run_config),
        "prompt_config": dict(prompt_config),
        "tool_defaults": dict(tool_defaults),
        "iteration": 0,
        "last_assistant_items": last_assistant_items,
        "last_function_call_items": last_function_call_items,
        "last_tool_outputs": last_tool_outputs,
    }


def run_one_iteration(state: Dict[str, Any]) -> bool:
    """Run one tool-call iteration. Returns False when investigator produced a final answer."""
    response = state["response"]
    tool_calls = list(ial._iter_function_calls(response))
    state["iteration"] += 1
    iteration = state["iteration"]

    if not tool_calls:
        final_text = state.get("cleaned_response_text", "")
        output_path = state["run_dir"] / "final_investigator_output.txt"
        if final_text:
            output_path.write_text(final_text + "\n", encoding="utf-8")
            print(final_text)
        else:
            output_path.write_text("[no output text returned]\n", encoding="utf-8")
            print("[no output text returned]")
        return False

    tool_outputs: List[Dict[str, Any]] = []
    current_function_call_items = ial._function_call_items(response)
    defaults = state["tool_defaults"]

    for call in tool_calls:
        name = ial._get_attr(call, "name")
        call_id = ial._get_attr(call, "call_id")
        arguments = ial._get_attr(call, "arguments", "{}")
        raw_arguments = arguments

        if isinstance(arguments, str):
            try:
                args_dict = json.loads(arguments)
            except json.JSONDecodeError:
                args_dict = {}
        else:
            args_dict = dict(arguments)

        if name == "generate_and_query":
            probe_run_dir = ial._make_indexed_child_dir(state["probe_runs_root"])
            call_cfg = ial.ToolConfig(
                aux_model=state["run_config"]["aux_model"],
                context_output_root=probe_run_dir / "generated_contexts",
                target_output_root=probe_run_dir / "role_samples",
            )

            effective_generate_args = {
                "hint": args_dict.get("hint", ""),
                "k": int(args_dict.get("k", defaults["k"])),
                "out_dir": "generated_contexts",
                "validate": bool(args_dict.get("validate", defaults["validate"])),
                "model": args_dict.get("model"),
                "target_role": state["run_config"]["sample_role"],
            }
            gen_result = ial.generate_contexts_tool(
                state["client"],
                call_cfg,
                hint=effective_generate_args["hint"],
                k=effective_generate_args["k"],
                out_dir=effective_generate_args["out_dir"],
                validate=effective_generate_args["validate"],
                model=effective_generate_args["model"],
                target_role=effective_generate_args["target_role"],
            )

            contexts_path = [gen_result.get("out_dir")] if gen_result.get("out_dir") else []
            effective_query_args = {
                "contexts_path": contexts_path,
                "role": state["run_config"]["sample_role"],
                "out_dir": "role_samples",
                "samples_per_role": int(args_dict.get("samples_per_role", defaults["samples_per_role"])),
                "max_new_tokens": int(args_dict.get("max_new_tokens", defaults["max_new_tokens"])),
                "min_new_tokens": int(args_dict.get("min_new_tokens", defaults["min_new_tokens"])),
                "temperature": float(args_dict.get("temperature", defaults["temperature"])),
                "top_p": float(args_dict.get("top_p", defaults["top_p"])),
                "repetition_penalty": float(args_dict.get("repetition_penalty", defaults["repetition_penalty"])),
                "stop_on_eot": bool(args_dict.get("stop_on_eot", defaults["stop_on_eot"])),
                "max_items": int(args_dict.get("max_items", defaults["max_items"])),
                "max_chars": int(args_dict.get("max_chars", defaults["max_chars"])),
                "max_sample_chars": int(args_dict.get("max_sample_chars", defaults["max_sample_chars"])),
                "sample_index": int(args_dict.get("sample_index", defaults["sample_index"])),
            }
            query_result = ial.query_target_tool(call_cfg, **effective_query_args)

            _append_jsonl(
                state["tool_calls_path"],
                {
                    "iteration": iteration,
                    "call_id": call_id,
                    "name": name,
                    "arguments_raw": raw_arguments,
                    "arguments_parsed": args_dict,
                    "effective_generate_args": effective_generate_args,
                    "effective_query_args": effective_query_args,
                },
            )
            result = {"generate": gen_result, "query": query_result}
        else:
            _append_jsonl(
                state["tool_calls_path"],
                {
                    "iteration": iteration,
                    "call_id": call_id,
                    "name": name,
                    "arguments_raw": raw_arguments,
                    "arguments_parsed": args_dict,
                    "error": f"Unknown tool name: {name}",
                },
            )
            result = {"error": f"Unknown tool name: {name}"}

        tool_outputs.append(
            {
                "type": "function_call_output",
                "call_id": call_id,
                "output": json.dumps(result, ensure_ascii=False),
            }
        )

    # Keep only the executed function_call items + outputs so call/output pairs remain valid.
    state["last_function_call_items"] = current_function_call_items
    state["last_tool_outputs"] = tool_outputs

    context = ial.build_input_context(
        state["prompt_config"]["investigator_user_prompt"],
        ial._read_memory(state["memory_path"]),
        last_assistant_items=state["last_assistant_items"],
        last_function_call_items=state["last_function_call_items"],
        last_tool_outputs=state["last_tool_outputs"],
    )

    next_response = state["client"].responses.create(
        model=state["run_config"]["investigator_model"],
        instructions=state["prompt_config"]["system_prompt"],
        input=context,
        tools=state["tools"],
        tool_choice="auto",
        max_output_tokens=state["run_config"]["max_output_tokens"],
        parallel_tool_calls=state["run_config"]["parallel_tool_calls"],
    )

    _append_jsonl(state["history_path"], ial._response_to_loggable(next_response))
    response_text = ial._extract_output_text(next_response)
    cleaned_response_text, updates = ial.extract_memory_updates(response_text)
    ial.append_memory(
        state["memory_path"],
        updates,
        iteration,
        memory_events_path=state["memory_events_path"],
        model=state["run_config"]["investigator_model"],
    )

    state["response"] = next_response
    state["cleaned_response_text"] = cleaned_response_text
    state["last_assistant_items"] = ial._latest_assistant_message_items(next_response)

    print(f"Completed iteration {iteration}")
    return True


def run_iterations(state: Dict[str, Any], max_iterations: int = 3) -> None:
    for _ in range(max_iterations):
        keep_going = run_one_iteration(state)
        if not keep_going:
            break


def force_finalize(state: Dict[str, Any]) -> str:
    """Force final answer with tool_choice='none', same strategy as the script."""
    context = ial.build_input_context(
        state["prompt_config"]["investigator_user_prompt"],
        ial._read_memory(state["memory_path"]),
        last_assistant_items=state["last_assistant_items"],
        last_function_call_items=state["last_function_call_items"],
        last_tool_outputs=state["last_tool_outputs"],
        extra_user_message=(
            "Tool-call limit reached. Do not call tools. "
            "Provide your best final hypothesis now using the required format, "
            "based only on evidence collected so far."
        ),
    )

    final_response = state["client"].responses.create(
        model=state["run_config"]["investigator_model"],
        instructions=state["prompt_config"]["system_prompt"],
        input=context,
        tools=state["tools"],
        tool_choice="none",
        max_output_tokens=state["run_config"]["max_output_tokens"],
    )

    _append_jsonl(state["history_path"], ial._response_to_loggable(final_response))
    final_text_raw = ial._extract_output_text(final_response)
    final_text, updates = ial.extract_memory_updates(final_text_raw)
    ial.append_memory(
        state["memory_path"],
        updates,
        state["iteration"] + 1,
        memory_events_path=state["memory_events_path"],
        model=state["run_config"]["investigator_model"],
    )

    final_text = final_text or "[max iterations reached without final response]"
    (state["run_dir"] / "final_investigator_output.txt").write_text(final_text + "\n", encoding="utf-8")
    print(final_text)
    return final_text



In [ ]:
# Typical workflow:
# 1) Edit RUN_CONFIG/PROMPT_CONFIG/TOOL_DEFAULTS
# 2) Load target model once (Cell 3) and keep this kernel alive
# 3) Start a fresh session
# 4) Iterate interactively and compare prompt/max-iteration variants

session = start_session(RUN_CONFIG, PROMPT_CONFIG, TOOL_DEFAULTS)
print("Run dir:", session["run_dir"])
print("Memory path:", session["memory_path"])

# Run a few investigator iterations. Re-run for more.
run_iterations(session, max_iterations=3)

# Optional explicit finalization:
# force_finalize(session)

